# MSDL-JCI — Thesis Defense

**Title:** Multi-Source Deep Learning for JCI Direction Prediction
**Status:** Closed — rigorous negative result

Visualizations for thesis defense: architecture, training dynamics, classification, gating weights, trading simulation.

## Setup

In [11]:
from pathlib import Path
import sys

# Find project root by searching for pyproject.toml upward
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 150})
sns.set_style("whitegrid")

def save_fig(fig, name):
    out = ROOT / "reports" / name
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches="tight", dpi=150)
    print(f"Saved: {name}")
    plt.close(fig)


## 1. Architecture

In [12]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis("off")

mods = ["Technical\n(LSTM, 64)", "Macro\n(MLP, 16)", "News\n(BERT→64)"]
mod_colors = ["#4C72B0", "#55A868", "#C44E52"]

for i, (m, c) in enumerate(zip(mods, mod_colors)):
    x = (i - 1) * 3
    ax.add_patch(plt.Rectangle((x, 3), 2.5, 1.2, facecolor=c, edgecolor="black", linewidth=1.5))
    ax.text(x + 1.25, 3.6, m, ha="center", va="center", fontsize=9, fontweight="bold", color="white")

ax.add_patch(plt.Rectangle((3.5, 2.5), 3, 1.2, facecolor="#DD8888", edgecolor="black", linewidth=1.5))
ax.text(5, 3.1, "Soft Gating", ha="center", va="center", fontsize=9, fontweight="bold")
ax.add_patch(plt.Rectangle((4, 0.5), 2, 1.2, facecolor="#8888DD", edgecolor="black", linewidth=1.5))
ax.text(5, 1.1, "Fusion", ha="center", va="center", fontsize=9)

ax.set_xlim(-2, 7)
ax.set_ylim(0, 6)
ax.set_title("Adaptive Soft Gating Fusion Architecture", fontsize=14, fontweight="bold", pad=20)
save_fig(fig, "architecture_overview.png")


Saved: architecture_overview.png


## 2. Training Dynamics

In [13]:
train_log = pd.read_csv(ROOT / "reports" / "phase_06_training_dynamics" / "training_logs.csv")
train_log["sweep_val"] = pd.to_numeric(train_log["sweep"], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cfg in train_log["config"].unique():
    mask = train_log["config"] == cfg
    axes[0].plot(train_log.loc[mask, "sweep_val"], train_log.loc[mask, "val_auc"], "o-", label=f"config={cfg}")
    axes[1].plot(train_log.loc[mask, "sweep_val"], train_log.loc[mask, "val_mcc"], "o-", label=f"config={cfg}")

axes[0].set_xlabel("Hyperparameter Value")
axes[0].set_ylabel("Validation AUC")
axes[0].set_title("Val AUC vs LR")
axes[0].set_xscale("log")
axes[0].legend(fontsize=8)

axes[1].set_xlabel("Hyperparameter Value")
axes[1].set_ylabel("Validation MCC")
axes[1].set_title("Val MCC vs LR")
axes[1].set_xscale("log")
axes[1].legend(fontsize=8)

fig.suptitle("Training Dynamics: Hyperparameter Sweeps", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "training_dynamics.png")


Saved: training_dynamics.png


## 3. Classification Results

In [14]:
cls = pd.read_csv(ROOT / "data" / "processed" / "ablation_classification_metrics.csv")

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(cls))
width = 0.25

for i, col in enumerate(["Accuracy", "ROC-AUC", "F1-Score"]):
    vals = cls[col].values
    bars = ax.bar(x + i * width, vals, width, label=col)
    ax.bar_label(bars, fontsize=8, fmt="%.3f")

ax.set_xticks(x + width)
ax.set_xticklabels(cls["Model"], rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_title("Classification Metrics by Model")
ax.legend()
ax.set_ylim(0, 1.0)
save_fig(fig, "classification_results.png")


Saved: classification_results.png


## 4. Gating Weights

In [15]:
gw = pd.read_csv(ROOT / "data" / "processed" / "gating_weights_summary.csv")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(gw["Weight Parameter"], gw["Mean Weight"], color=["#4C72B0", "#55A868", "#C44E52"], edgecolor="black")
ax.bar_label(bars, fmt="%.3f", fontsize=11, fontweight="bold")
ax.set_ylabel("Mean Gating Weight")
ax.set_title("Average Soft Gating Weights")
ax.set_ylim(0, 1)
save_fig(fig, "gating_weights.png")


Saved: gating_weights.png


## 5. Trading Simulation

In [16]:
fin = pd.read_csv(ROOT / "data" / "processed" / "ablation_financial_metrics.csv")
gn = pd.read_csv(ROOT / "reports" / "phase_10_trading" / "gross_vs_net_results.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models = fin["Model"].tolist()
x = np.arange(len(models))

axes[0].bar(x - 0.175, fin["Total Return (%)"], 0.35, label="Total Return", color="#4C72B0")
axes[0].bar(x + 0.175, fin["Benchmark Return (%)"], 0.35, label="Buy-and-Hold", color="#AAAAAA")
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=30, ha="right", fontsize=8)
axes[0].set_ylabel("Return (%)")
axes[0].set_title("Strategy vs Buy-and-Hold")
axes[0].legend(fontsize=8)

bars = axes[1].bar(x, fin["Sharpe Ratio"], color="#55A868", edgecolor="black")
axes[1].bar_label(bars, fmt="%.3f", fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=30, ha="right", fontsize=8)
axes[1].set_ylabel("Sharpe Ratio")
axes[1].set_title("Annualized Sharpe Ratio")

fig.suptitle("Trading Simulation: Financial Performance", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "trading_financial.png")


Saved: trading_financial.png


## 6. Gross vs Net Returns

In [17]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(gn))

bars1 = ax.bar(x - 0.175, gn["gross"], 0.35, label="Gross Return", color="#4C72B0")
bars2 = ax.bar(x + 0.175, gn["net"], 0.35, label="Net Return", color="#C44E52")
ax.bar_label(bars1, fmt="%.2f%%", fontsize=9)
ax.bar_label(bars2, fmt="%.2f%%", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(gn["Model"], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Return (%)")
ax.set_title("Gross vs Net Returns (with transaction costs)")
ax.legend()

ax2 = ax.twinx()
ax2.plot(x, gn["cost_drag_pp"], "D-", color="black", label="Cost Drag (pp)", markersize=6)
ax2.set_ylabel("Cost Drag (percentage points)", fontsize=10)
ax2.legend(loc="upper left")

plt.tight_layout()
save_fig(fig, "gross_vs_net.png")


Saved: gross_vs_net.png


## 7. Key Findings

In [18]:
summary_data = {
    "Finding": [
        "Soft Gating collapsed (MCC ≈ 0)",
        "Branch AUC ≈ 0.50 (no signal)",
        "Gating balanced but not informative",
        "Best heuristic: Momentum-5d",
        "Conclusion: Negative result"
    ],
    "Status": [
        "Not viable",
        "No predictive edge",
        "Weight != Signal",
        "Needs validation",
        "Closed"
    ]
}
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))


                            Finding             Status
    Soft Gating collapsed (MCC ≈ 0)         Not viable
      Branch AUC ≈ 0.50 (no signal) No predictive edge
Gating balanced but not informative   Weight != Signal
        Best heuristic: Momentum-5d   Needs validation
        Conclusion: Negative result             Closed
